# 01 - Data Exploration

Exploratory Data Analysis (EDA) for PRISM project data.

## Objectives
- Load and inspect project data
- Understand data distributions
- Identify data quality issues
- Discover patterns and correlations
- Analyze text fields for LLM processing

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Libraries loaded successfully!")

## 1. Load Data

In [ ]:
# Load project data from data/raw
data_path = Path('../data/raw/sample_projects.csv')
df = pd.read_csv(data_path)

print(f"Loaded {len(df)} projects with {len(df.columns)} columns")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
# Preview data
df.head()

## 2. Data Overview

In [ ]:
# Basic info
print("=" * 60)
print("DATA OVERVIEW")
print("=" * 60)
print(f"\nShape: {df.shape}")
print(f"\nColumn Types:")
print(df.dtypes)

In [ ]:
# Missing values analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)

print("\nMissing Values:")
if len(missing_df) > 0:
    print(missing_df)
else:
    print("No missing values!")

## 3. Numerical Features Analysis

In [ ]:
# Define numerical columns
numerical_cols = [
    'budget', 'spent', 'planned_hours', 'actual_hours',
    'team_size', 'completion_rate', 'complexity_score',
    'dependencies', 'velocity', 'defect_rate', 'team_turnover'
]

# Filter to available columns
numerical_cols = [c for c in numerical_cols if c in df.columns]

# Summary statistics
df[numerical_cols].describe()

In [ ]:
# Distribution plots for numerical features
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    if i < len(axes):
        ax = axes[i]
        df[col].hist(ax=ax, bins=15, edgecolor='black', alpha=0.7)
        ax.set_title(col, fontsize=10)
        ax.set_xlabel('')

# Hide empty subplots
for j in range(len(numerical_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.suptitle('Numerical Feature Distributions', y=1.02, fontsize=14)
plt.show()

In [ ]:
# Budget vs Spent analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Budget vs Spent scatter
colors = {'High': 'red', 'Medium': 'orange', 'Low': 'green'}
for risk in df['risk_level'].unique():
    mask = df['risk_level'] == risk
    axes[0].scatter(df.loc[mask, 'budget'], df.loc[mask, 'spent'], 
                   label=risk, alpha=0.7, c=colors.get(risk, 'gray'), s=100)

# Add diagonal line (budget = spent)
max_val = max(df['budget'].max(), df['spent'].max())
axes[0].plot([0, max_val], [0, max_val], 'k--', alpha=0.5, label='Budget = Spent')
axes[0].set_xlabel('Budget ($)')
axes[0].set_ylabel('Spent ($)')
axes[0].set_title('Budget vs Spent by Risk Level')
axes[0].legend()

# Budget utilization
df['budget_utilization'] = (df['spent'] / df['budget'] * 100).round(2)
df['budget_utilization'].hist(ax=axes[1], bins=15, edgecolor='black', alpha=0.7)
axes[1].axvline(x=100, color='red', linestyle='--', label='100% (On Budget)')
axes[1].set_xlabel('Budget Utilization (%)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Budget Utilization Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Categorical Features Analysis

In [ ]:
# Define categorical columns
categorical_cols = ['status', 'priority', 'risk_level', 'methodology', 
                   'department', 'client_type', 'project_type']

# Filter to available columns
categorical_cols = [c for c in categorical_cols if c in df.columns]

# Value counts for each
for col in categorical_cols:
    print(f"\n{'='*40}")
    print(f"{col.upper()}")
    print(f"{'='*40}")
    print(df[col].value_counts())

In [ ]:
# Categorical distribution plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols[:6]):
    ax = axes[i]
    counts = df[col].value_counts()
    
    # Use specific colors for risk_level
    if col == 'risk_level':
        color_map = {'High': 'red', 'Medium': 'orange', 'Low': 'green'}
        bar_colors = [color_map.get(x, 'gray') for x in counts.index]
    else:
        bar_colors = plt.cm.Set2(range(len(counts)))
    
    counts.plot(kind='bar', ax=ax, color=bar_colors, edgecolor='black')
    ax.set_title(col, fontsize=12)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.suptitle('Categorical Feature Distributions', y=1.02, fontsize=14)
plt.show()

## 5. Risk Level Analysis (Target Variable)

In [ ]:
# Risk level distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
risk_counts = df['risk_level'].value_counts()
colors = ['red', 'orange', 'green']
axes[0].pie(risk_counts, labels=risk_counts.index, autopct='%1.1f%%', 
           colors=colors, startangle=90, explode=[0.05]*len(risk_counts))
axes[0].set_title('Risk Level Distribution')

# Risk by priority
cross_tab = pd.crosstab(df['priority'], df['risk_level'])
cross_tab.plot(kind='bar', ax=axes[1], color=['green', 'orange', 'red'], edgecolor='black')
axes[1].set_title('Risk Level by Priority')
axes[1].set_xlabel('Priority')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Risk Level')

plt.tight_layout()
plt.show()

In [ ]:
# Numerical features by risk level
risk_summary = df.groupby('risk_level')[numerical_cols].mean().round(2)
risk_summary

In [ ]:
# Box plots by risk level for key metrics
key_metrics = ['completion_rate', 'defect_rate', 'team_turnover', 'complexity_score']
key_metrics = [c for c in key_metrics if c in df.columns]

fig, axes = plt.subplots(1, len(key_metrics), figsize=(16, 5))

for i, col in enumerate(key_metrics):
    order = ['Low', 'Medium', 'High']
    sns.boxplot(data=df, x='risk_level', y=col, ax=axes[i], order=order,
               palette={'Low': 'green', 'Medium': 'orange', 'High': 'red'})
    axes[i].set_title(col)
    axes[i].set_xlabel('Risk Level')

plt.tight_layout()
plt.suptitle('Key Metrics by Risk Level', y=1.02, fontsize=14)
plt.show()

## 6. Correlation Analysis

In [ ]:
# Correlation matrix
corr_cols = [c for c in numerical_cols if df[c].notna().sum() > 0]
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
           cmap='RdYlGn_r', center=0, vmin=-1, vmax=1,
           square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Key correlations with numeric risk level
df['risk_numeric'] = df['risk_level'].map({'Low': 0, 'Medium': 1, 'High': 2})

risk_corr = df[corr_cols + ['risk_numeric']].corr()['risk_numeric'].drop('risk_numeric').sort_values(ascending=False)

print("Correlations with Risk Level:")
print("=" * 40)
print(risk_corr.round(3))

## 7. Text Fields Analysis (for LLM)

In [ ]:
# Text columns
text_cols = ['status_comments', 'project_description', 'team_feedback']
text_cols = [c for c in text_cols if c in df.columns]

# Text length analysis
for col in text_cols:
    df[f'{col}_length'] = df[col].fillna('').str.len()
    print(f"\n{col}:")
    print(f"  - Mean length: {df[f'{col}_length'].mean():.0f} chars")
    print(f"  - Min length: {df[f'{col}_length'].min():.0f} chars")
    print(f"  - Max length: {df[f'{col}_length'].max():.0f} chars")

In [ ]:
# Sample text by risk level
print("\n" + "=" * 60)
print("SAMPLE STATUS COMMENTS BY RISK LEVEL")
print("=" * 60)

for risk in ['High', 'Medium', 'Low']:
    sample = df[df['risk_level'] == risk]['status_comments'].iloc[0] if len(df[df['risk_level'] == risk]) > 0 else 'N/A'
    print(f"\n[{risk} Risk]:")
    print(f"{sample[:300]}..." if len(sample) > 300 else sample)

In [ ]:
# Word frequency analysis (simple)
from collections import Counter
import re

def get_word_freq(text_series, min_length=4):
    """Get word frequency from text series."""
    all_text = ' '.join(text_series.fillna('').str.lower())
    words = re.findall(r'\b[a-z]+\b', all_text)
    words = [w for w in words if len(w) >= min_length]
    return Counter(words)

# Common words in high-risk vs low-risk projects
high_risk_words = get_word_freq(df[df['risk_level'] == 'High']['status_comments'])
low_risk_words = get_word_freq(df[df['risk_level'] == 'Low']['status_comments'])

print("\nTop 10 Words in HIGH-Risk Projects:")
print(high_risk_words.most_common(10))

print("\nTop 10 Words in LOW-Risk Projects:")
print(low_risk_words.most_common(10))

## 8. Data Quality Summary

In [ ]:
# Data quality report
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

print(f"\n✅ Total projects: {len(df)}")
print(f"✅ Total features: {len(df.columns)}")
print(f"✅ Missing values: {df.isnull().sum().sum()} ({(df.isnull().sum().sum() / df.size * 100):.2f}%)")

print(f"\n📊 Risk Distribution:")
for risk, count in df['risk_level'].value_counts().items():
    pct = count / len(df) * 100
    print(f"   - {risk}: {count} ({pct:.1f}%)")

print(f"\n📝 Text Data Available:")
for col in text_cols:
    non_empty = (df[col].notna() & (df[col].str.len() > 0)).sum()
    print(f"   - {col}: {non_empty}/{len(df)} projects")

print(f"\n⚠️ Potential Issues:")
# Check for outliers
over_budget = (df['spent'] > df['budget']).sum()
print(f"   - Projects over budget: {over_budget}")
over_hours = (df['actual_hours'] > df['planned_hours']).sum()
print(f"   - Projects over planned hours: {over_hours}")

## 9. Key Insights

### Findings:
1. **Risk Distribution**: The dataset shows a mix of risk levels suitable for ML training
2. **Budget Patterns**: Some projects are over budget, which correlates with higher risk
3. **Team Factors**: Team turnover and defect rates appear to correlate with risk level
4. **Text Data**: Rich status comments available for LLM sentiment analysis

### Next Steps:
1. Feature engineering (see `02_feature_engineering.ipynb`)
2. ML model training (see `03_ml_modeling.ipynb`)
3. LLM analysis of text fields